In [1]:
import pandas as pd

df = pd.read_csv("s3://capstone-project-mcallister/teams.csv")

In [ ]:
import pandas as pd
import numpy as np

# Create a fake dataframe with 10 records
np.random.seed(42)
df = pd.DataFrame({
    'feature_1': np.random.rand(10),
    'feature_2': np.random.rand(10),
    'feature_3': np.random.randint(0, 100, 10),
    'feature_4': np.random.choice(['A', 'B', 'C'], 10),
    'target': np.random.randint(0, 2, 10)
})

print("Created fake dataframe:")
print(df)

In [5]:
df.head()

,id,school,conference,division,color,logos,latitude_school,longitude_school,test_col
0,2005,Air Force,Mountain West,NaN,#004a7b,http://a.espncdn.com/i/teamlogos/ncaa/500/2005...,38.996970,-104.843616,1
1,2006,Akron,Mid-American,NaN,#00285e,http://a.espncdn.com/i/teamlogos/ncaa/500/2006...,41.072553,-81.508341,1
2,333,Alabama,SEC,NaN,#9e1632,http://a.espncdn.com/i/teamlogos/ncaa/500/333.png,33.208275,-87.550384,1
3,2026,App State,Sun Belt,East,NaN,http://a.espncdn.com/i/teamlogos/ncaa/500/2026...,36.211427,-81.685428,1
4,12,Arizona,Big 12,NaN,#0c234b,http://a.espncdn.com/i/teamlogos/ncaa/500/12.png,32.228805,-110.948868,1


In [3]:
df['test_col'] = 1

In [4]:
df.to_csv("s3://capstone-project-mcallister/team_edited.csv", index=False)

In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# Create a fake dataframe with 10 records
np.random.seed(0)
df = pd.DataFrame({
    'feature_1': np.random.rand(10),
    'feature_2': np.random.rand(10),
    'feature_3': np.random.randint(0, 100, 10),
    'feature_4': np.random.choice(['A', 'B', 'C'], 10),
    'target': np.random.randint(0, 2, 10)
})

# Split out target and features
X = df.drop('target', axis=1)
y = df['target']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train the dummy classifier
clf = DummyClassifier(strategy='most_frequent')
clf.fit(X_train, y_train)

# Evaluate
score = clf.score(X_test, y_test)
print(f"Dummy Classifier Accuracy: {score:.4f}")

Created fake dataframe:
   feature_1  feature_2  feature_3 feature_4  target
0   0.374540   0.020584         58         C       1
1   0.950714   0.969910         41         C       1
2   0.731994   0.832443         91         A       0
3   0.598658   0.212339         59         C       1
4   0.156019   0.181825         79         A       1
5   0.155995   0.183405         14         C       1
6   0.058084   0.304242         61         C       1
7   0.866176   0.524756         61         A       0
8   0.601115   0.431945         46         A       1
9   0.708073   0.291229         61         C       0
Dummy Classifier Accuracy: 1.0000


In [10]:
import pickle
import boto3

# Save model locally to disk
pickle.dump(clf, open('dummy_classifier.pkl', 'wb'))

# Upload to S3
s3_client = boto3.client('s3')
s3_client.upload_file(
    'dummy_classifier.pkl',
    'capstone-project-mcallister',
    'dummy_classifier.pkl'
)

print("Dummy classifier saved to S3!")

Dummy classifier saved to S3!


In [11]:
import boto3
import pickle
import io

# Create S3 client
s3_client = boto3.client('s3')

# Download the model from S3
response = s3_client.get_object(
    Bucket='capstone-project-mcallister',
    Key='dummy_classifier.pkl'
)

# Load the model from the response
model_buffer = io.BytesIO(response['Body'].read())
clf = pickle.load(model_buffer)